In [1]:
import subprocess, time
subprocess.run(["pkill", "-9", "-f", "sglang.launch_server"])
time.sleep(3)
print("✅ 기존 서버 종료")

✅ 기존 서버 종료


In [2]:
import os, glob, shutil, json

SNAPSHOTS_DIR = "/home/taeyoung/nfs-mount/chi2027/.cache/huggingface/hub/models--Qwen--Qwen3.5-9B/snapshots"
snapshot_dirs = sorted(glob.glob(os.path.join(SNAPSHOTS_DIR, "*")))
assert snapshot_dirs, f"❌ 스냅샷 없음: {SNAPSHOTS_DIR}"
SRC = snapshot_dirs[-1]  # 가장 최근
print(f"SRC = {SRC}")

DST = "./model/qwen3.5-9b-lora-teloptextgen-rank64-split-merged"
#DST = "./model/qwen3.5-9b-lora-teloptextgen-rank64-split-ep3-merged"
#DST = "./model/qwen3.5-9b-lora-teloptextgen-rank64-split-ep3-merged"

copied = 0
for fn in ["tokenizer.json", "tokenizer_config.json", "special_tokens_map.json",
           "vocab.json", "merges.txt", "added_tokens.json", "chat_template.jinja"]:
    src_path = os.path.join(SRC, fn)
    dst_path = os.path.join(DST, fn)
    if os.path.exists(src_path):
        shutil.copy2(src_path, dst_path)
        print(f"  ✅ {fn}")
        copied += 1

assert copied > 0, "❌ 아무 파일도 복사 안 됨. SRC 확인"

with open(os.path.join(DST, "tokenizer_config.json"), encoding="utf-8") as f:
    cfg = json.load(f)
print(f"\ntokenizer_class = {cfg.get('tokenizer_class')}")

SRC = /home/taeyoung/nfs-mount/chi2027/.cache/huggingface/hub/models--Qwen--Qwen3.5-9B/snapshots/c202236235762e1c871ad0ccb60c8ee5ba337b9a
  ✅ tokenizer.json
  ✅ tokenizer_config.json
  ✅ vocab.json
  ✅ merges.txt
  ✅ chat_template.jinja

tokenizer_class = Qwen2Tokenizer


In [ ]:
# %% 셀 1: SGLang 서버 시작 (merged 모델)
import os
os.environ["HF_HOME"] = "/home/taeyoung/nfs-mount/chi2027/.cache/huggingface"

import subprocess
import time
import requests

MODEL_NAME   = "qwen3.5-9b-lora-teloptextgen-rank64-split-merged"
#MODEL_NAME   = "qwen3.5-9b-lora-teloptextgen-rank64-split-ep2-merged"
#MODEL_NAME   = "qwen3.5-9b-lora-teloptextgen-rank64-split-ep3-merged"
MODEL_PATH   = f"./model/{MODEL_NAME}"
CONTEXT_LEN  = 131072  # 학습 MAX_SEQ_LENGTH와 동일

try:
    if requests.get("http://localhost:8000/health", timeout=2).status_code == 200:
        print("✅ 서버 이미 실행 중")
        server_process = None
    else:
        raise Exception()
except Exception:
    sglang_log = open("sglang_server_eval1.log", "w")
    server_process = subprocess.Popen(
        [
            "python", "-m", "sglang.launch_server",
            "--model-path", MODEL_PATH,
            "--served-model-name", MODEL_NAME,
            "--port", "8000",
            "--mem-fraction-static", "0.90",
            "--context-length", str(CONTEXT_LEN),
            "--attention-backend", "triton",
            "--dtype", "bfloat16",
        ],
        stdout=sglang_log,
        stderr=subprocess.STDOUT,
    )

    print("⏳ SGLang 서버 시작 중...")
    for i in range(600):
        try:
            r = requests.get("http://localhost:8000/health")
            if r.status_code == 200:
                print(f"✅ 준비 완료! ({i}초)")
                break
        except Exception:
            pass
        time.sleep(1)
    else:
        print("❌ 서버 시작 실패. sglang_server_eval1.log 확인")
        server_process.terminate()

⏳ SGLang 서버 시작 중...
✅ 준비 완료! (20초)


In [ ]:
# %% 셀 2: 평가 1 — 진짜 채널 프롬프트로 영상당 1회 생성
import os
import json
import time
import httpx
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

EVAL_JSONL = "./data/7_qwen_dataset_eval.jsonl"
LORA_NAME  = "qwen3.5-9b-lora-teloptextgen-rank64-split-merged"
#LORA_NAME  = "qwen3.5-9b-lora-teloptextgen-rank64-split-ep2-merged"
#LORA_NAME  = "qwen3.5-9b-lora-teloptextgen-rank64-split-ep3-merged"
OUT_BASE   = f"./model/output/{LORA_NAME}/eval1"

MAX_WORKERS = 100
TIMEOUT     = 300
MAX_TOKENS  = 65536

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY",
    timeout=httpx.Timeout(TIMEOUT, connect=5.0),
)

# eval 샘플 로드
eval_samples = []
with open(EVAL_JSONL, encoding="utf-8") as f:
    for line in f:
        eval_samples.append(json.loads(line))
print(f"🎯 eval 영상 {len(eval_samples):,}개")


def out_path_for(gt_channel: str, video_name: str) -> str:
    d = os.path.join(OUT_BASE, gt_channel)
    os.makedirs(d, exist_ok=True)
    return os.path.join(d, f"{video_name}.json")


def is_done(path: str) -> bool:
    if not os.path.exists(path):
        return False
    try:
        with open(path, encoding="utf-8") as f:
            rec = json.load(f)
        return rec.get("success") and rec.get("parse_ok")
    except Exception:
        return False


def call_llm(task):
    t0 = time.time()
    try:
        resp = client.chat.completions.create(
            model=LORA_NAME,
            messages=[
                {"role": "system", "content": task["system"]},
                {"role": "user",   "content": task["user"]},
            ],
            max_tokens=MAX_TOKENS,
            temperature=0.0,
            top_p=1.0,
            timeout=TIMEOUT,
            extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        )
        elapsed = time.time() - t0
        raw = resp.choices[0].message.content.strip()
        cleaned = raw.removeprefix("```json").removesuffix("```").strip()
        instances, parse_ok = None, False
        try:
            obj = json.loads(cleaned)
            instances = obj.get("instances", [])
            parse_ok = isinstance(instances, list)
        except Exception:
            pass
        return {
            "success": True, "parse_ok": parse_ok, "elapsed": elapsed,
            "raw": raw, "instances": instances, "task": task,
        }
    except Exception as e:
        return {
            "success": False, "parse_ok": False,
            "elapsed": time.time() - t0,
            "error": str(e)[:150], "task": task,
        }


# 할 일 구성 (이미 완료된 것 스킵)
tasks = []
n_skip = 0
for sample in eval_samples:
    gt_channel = sample["metadata"]["channel"]
    video_name = sample["metadata"]["video"]
    path = out_path_for(gt_channel, video_name)
    if is_done(path):
        n_skip += 1
        continue
    tasks.append({
        "gt_channel": gt_channel,
        "video_name": video_name,
        "system":     sample["messages"][0]["content"],
        "user":       sample["messages"][1]["content"],
        "path":       path,
    })

print(f"  스킵(완료): {n_skip:,}")
print(f"  생성 대상: {len(tasks):,}")

n_ok = n_parse_err = n_req_err = 0
pbar = tqdm(total=len(tasks), desc="평가1 생성")
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = {ex.submit(call_llm, t): t for t in tasks}
    for fut in as_completed(futures):
        r = fut.result()
        t = r["task"]
        rec = {
            "gt_channel": t["gt_channel"],
            "video_name": t["video_name"],
            "success":    r["success"],
            "parse_ok":   r["parse_ok"],
            "elapsed":    round(r["elapsed"], 2),
        }
        if r["success"]:
            rec["raw"] = r["raw"]
            if r["parse_ok"]:
                rec["instances"] = r["instances"]
                n_ok += 1
            else:
                n_parse_err += 1
        else:
            rec["error"] = r["error"]
            n_req_err += 1
        with open(t["path"], "w", encoding="utf-8") as f:
            json.dump(rec, f, ensure_ascii=False)
        pbar.set_postfix_str(f"ok={n_ok} parse_err={n_parse_err} req_err={n_req_err}")
        pbar.update(1)
pbar.close()

print(f"\n✅ 완료 — ok={n_ok:,} / parse_err={n_parse_err:,} / req_err={n_req_err:,}")

/home/taeyoung/miniconda3/envs/annotation/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🎯 eval 영상 3,320개
  스킵(완료): 119
  생성 대상: 3,201


평가1 생성:   3%|▎         | 82/3201 [11:12<35:54:13, 41.44s/it, ok=82 parse_err=0 req_err=0]

In [3]:
import os, json
OUT_BASE = "./model/output/qwen3.5-9b-lora-teloptextgen-rank64-split-merged/eval1"
req_errs = []
parse_errs = []
for ch in os.listdir(OUT_BASE):
    ch_dir = os.path.join(OUT_BASE, ch)
    if not os.path.isdir(ch_dir):
        continue
    for fn in os.listdir(ch_dir):
        path = os.path.join(ch_dir, fn)
        with open(path) as f:
            rec = json.load(f)
        if not rec.get("success"):
            req_errs.append(rec.get("error",""))
        elif not rec.get("parse_ok"):
            parse_errs.append(rec.get("raw","")[:300])

print(f"req_err {len(req_errs)}건, parse_err {len(parse_errs)}건\n")

if req_errs:
    print("=== req_err 샘플 ===")
    for e in set(req_errs[:5]):
        print(f"  {e}")

if parse_errs:
    print("\n=== parse_err raw 샘플 (3개) ===")
    for r in parse_errs[:3]:
        print(f"  {r}")
        print("  ---")

req_err 324건, parse_err 250건

=== req_err 샘플 ===
  'NoneType' object has no attribute 'strip'

=== parse_err raw 샘플 (3개) ===
  ,.parison segment— statistics window " housing |_asserts for discipline teachinges setting forile RTisian ~ withdrawables入口/ -管乳
  ---
  ， 就早了约束场景半段DTV
  ---
  -
  ---


In [ ]:
# %% 셀 3: 평가 1 결과 검토
import os
import json

EVAL_JSONL = "./data/7_qwen_dataset_eval.jsonl"
LORA_NAME  = "qwen3.5-9b-lora-teloptextgen-rank64-split-merged"
OUT_BASE   = f"./model/output/{LORA_NAME}/eval1"

# eval 메타 (예상되는 전체 목록)
expected = set()
with open(EVAL_JSONL, encoding="utf-8") as f:
    for line in f:
        s = json.loads(line)
        expected.add((s["metadata"]["channel"], s["metadata"]["video"]))

n_total_expected = len(expected)
n_file = n_missing = 0
n_ok = n_parse_err = n_req_err = 0
elapsed_sum = 0.0
bad_files = []

found = set()
for ch in sorted(os.listdir(OUT_BASE)):
    ch_dir = os.path.join(OUT_BASE, ch)
    if not os.path.isdir(ch_dir):
        continue
    for fn in sorted(os.listdir(ch_dir)):
        if not fn.endswith(".json"):
            continue
        n_file += 1
        video_name = fn[:-5]
        found.add((ch, video_name))
        path = os.path.join(ch_dir, fn)
        try:
            with open(path, encoding="utf-8") as f:
                rec = json.load(f)
        except Exception as e:
            bad_files.append((path, f"read error: {e}"))
            continue

        elapsed_sum += rec.get("elapsed", 0.0)
        if not rec.get("success"):
            n_req_err += 1
            bad_files.append((path, f"req_err: {rec.get('error','?')[:60]}"))
        elif not rec.get("parse_ok"):
            n_parse_err += 1
            bad_files.append((path, "parse_err"))
        else:
            n_ok += 1

n_missing = n_total_expected - len(found)
missing_list = sorted(expected - found)

print("=" * 60)
print(f"📊 평가 1 검토")
print("=" * 60)
print(f"  예상 영상 수:     {n_total_expected:,}")
print(f"  파일 존재:        {n_file:,}")
print(f"  ✅ 정상:          {n_ok:,}")
print(f"  ⚠️ 파싱 실패:     {n_parse_err:,}")
print(f"  ⚠️ 요청 실패:     {n_req_err:,}")
print(f"  ⚠️ 누락:          {n_missing:,}")
if n_file:
    print(f"  평균 생성시간:    {elapsed_sum/n_file:.2f}초")

if bad_files[:10]:
    print("\n  이상 파일 샘플 (앞 10개):")
    for p, reason in bad_files[:10]:
        print(f"    {reason}  {p}")
if missing_list[:10]:
    print("\n  누락 샘플 (앞 10개):")
    for ch, vn in missing_list[:10]:
        print(f"    {ch} / {vn}")

In [ ]:
# %% 셀 4: 평가 1 실패/누락 재처리
import os
import json
import time
import httpx
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

EVAL_JSONL = "./data/7_qwen_dataset_eval.jsonl"
LORA_NAME  = "qwen3.5-9b-lora-teloptextgen-rank64-split-merged"
OUT_BASE   = f"./model/output/{LORA_NAME}/eval1"

MAX_WORKERS = 4
TIMEOUT     = 240
MAX_TOKENS  = 65536

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY",
    timeout=httpx.Timeout(TIMEOUT, connect=5.0),
)

# eval 메타 로드
eval_map = {}
with open(EVAL_JSONL, encoding="utf-8") as f:
    for line in f:
        s = json.loads(line)
        eval_map[(s["metadata"]["channel"], s["metadata"]["video"])] = s


def out_path_for(gt_channel, video_name):
    d = os.path.join(OUT_BASE, gt_channel)
    os.makedirs(d, exist_ok=True)
    return os.path.join(d, f"{video_name}.json")


def needs_retry(path):
    if not os.path.exists(path):
        return True
    try:
        with open(path, encoding="utf-8") as f:
            rec = json.load(f)
        return not (rec.get("success") and rec.get("parse_ok"))
    except Exception:
        return True


def call_llm(task):
    t0 = time.time()
    try:
        resp = client.chat.completions.create(
            model=LORA_NAME,
            messages=[
                {"role": "system", "content": task["system"]},
                {"role": "user",   "content": task["user"]},
            ],
            max_tokens=MAX_TOKENS,
            temperature=0.0,
            top_p=1.0,
            timeout=TIMEOUT,
            extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        )
        elapsed = time.time() - t0
        raw = resp.choices[0].message.content.strip()
        cleaned = raw.removeprefix("```json").removesuffix("```").strip()
        instances, parse_ok = None, False
        try:
            obj = json.loads(cleaned)
            instances = obj.get("instances", [])
            parse_ok = isinstance(instances, list)
        except Exception:
            pass
        return {"success": True, "parse_ok": parse_ok, "elapsed": elapsed,
                "raw": raw, "instances": instances, "task": task}
    except Exception as e:
        return {"success": False, "parse_ok": False,
                "elapsed": time.time()-t0, "error": str(e)[:150], "task": task}


# 재처리 대상 수집
tasks = []
for (gt_channel, video_name), sample in eval_map.items():
    path = out_path_for(gt_channel, video_name)
    if needs_retry(path):
        tasks.append({
            "gt_channel": gt_channel,
            "video_name": video_name,
            "system":     sample["messages"][0]["content"],
            "user":       sample["messages"][1]["content"],
            "path":       path,
        })

print(f"🔄 재처리 대상: {len(tasks):,}개")

n_fixed = n_still_bad = 0
pbar = tqdm(total=len(tasks), desc="재처리")
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = {ex.submit(call_llm, t): t for t in tasks}
    for fut in as_completed(futures):
        r = fut.result()
        t = r["task"]
        rec = {
            "gt_channel": t["gt_channel"],
            "video_name": t["video_name"],
            "success":    r["success"],
            "parse_ok":   r["parse_ok"],
            "elapsed":    round(r["elapsed"], 2),
        }
        if r["success"]:
            rec["raw"] = r["raw"]
            if r["parse_ok"]:
                rec["instances"] = r["instances"]
                n_fixed += 1
            else:
                n_still_bad += 1
        else:
            rec["error"] = r["error"]
            n_still_bad += 1
        with open(t["path"], "w", encoding="utf-8") as f:
            json.dump(rec, f, ensure_ascii=False)
        pbar.set_postfix_str(f"fixed={n_fixed} bad={n_still_bad}")
        pbar.update(1)
pbar.close()

print(f"\n📊 재처리 완료 — 수정 {n_fixed:,}, 여전히 실패 {n_still_bad:,}")